# WAAM 로봇 3대 경로 계획 — 코랩에서 풀고 계산 시간 재기

INE4107 IC-PBL. 공개 형상 6개(또는 연습 3 실제 부품 모양 21개, 원하는 시나리오 폴더)를 우리 알고리즘(`src/solve.py`)으로 풀고,
형상마다 **계산 시간**을 잰 뒤 교수님 채점기(WAAM Validator)로 채점합니다.

- **런타임:** 무료 일반 CPU (메뉴 *런타임 → 런타임 유형 변경 → CPU*). GPU·TPU 는 쓰지 않습니다.
- **계산 시간:** 형상 폴더를 넣은 순간부터 `trajectory.csv` 가 나올 때까지(형상 하나당). 설치·채점 시간은 넣지 않습니다.
  미리 학습하는 단계는 없습니다.
- **확정적:** 무작위를 쓰지 않아 같은 환경에서는 몇 번을 돌려도 같은 궤적이 나옵니다.
- 위에서부터 차례로 실행하고, 4번 표와 1번 컴퓨터 정보를 캡처하면 됩니다(5번).
  6번은 대시보드에서 노트북 측정과 나란히 볼 기록 파일을 내려받습니다.

## 1. 컴퓨터 정보

In [ ]:
!echo "CPU 개수: $(nproc)"
!lscpu | grep -E "Model name|^CPU\(s\)"
!free -g | head -2
!python --version

## 2. 설치 (계산 시간에 넣지 않음)
우리 저장소와 채점기를 내려받고 채점기를 설치합니다. 1–2분 걸립니다.

In [ ]:
%%bash
cd /content
[ -d waam-icpbl ] || git clone -q https://github.com/hosukim03/waam-icpbl.git
cd waam-icpbl && git pull -q
[ -d WAAM_Validator ] || git clone -q https://github.com/Ha-An/WAAM_Validator.git
pip install -q -e WAAM_Validator
git log -1 --format='우리 코드: %h (%ci)'
git -C WAAM_Validator log -1 --format='채점기: %h (%ci)'

## 3. 풀기 — 형상마다 계산 시간
코드 오른쪽 칸(`SET`)에서 묶음을 고릅니다.
- **공개 6개**: 채점기에 들어 있는 공개 형상(기본).
- **연습 3 (실제 부품 21개)**: 형상 파일이 저장소에 없어서 우리 생성기(`tools/make_jobs_parts.py`, 시드 1126)로
  먼저 만듭니다(1분 안팎, 계산 시간에 넣지 않음). 노트북에서 잰 형상과 똑같은지 지문으로 확인합니다.

다른 형상을 풀려면 폴더를 올리고(왼쪽 파일 창) 아래 `SCENARIOS` 에 그 경로를 넣으면 됩니다.

In [ ]:
import hashlib, re, subprocess, sys, time
from pathlib import Path

SET = '공개 6개'  #@param ['공개 6개', '연습 3 (실제 부품 21개)']
ROOT = Path('/content/waam-icpbl')
OUT = Path('/content/out')
LIMIT_S = 300.0              # 5분
P3_FP = '1689b785a95d'       # 노트북에서 잰 연습 3 형상(data/parts1126)의 지문


def fingerprint(d):
    # 형상마다 config.yaml(줄바꿈은 맞춰서)과 target.stl 을 이어 붙인 md5 앞 12자리
    h = hashlib.md5()
    for s in sorted(p for p in d.iterdir() if (p / 'target.stl').exists()):
        h.update(s.name.encode())
        h.update((s / 'config.yaml').read_bytes().replace(b'\r\n', b'\n'))
        h.update((s / 'target.stl').read_bytes())
    return h.hexdigest()[:12]


if SET.startswith('공개'):
    SCENARIOS = [ROOT / 'WAAM_Validator' / 'tests' / m for m in ['01', '02', '03', '04', '05', '06']]
else:
    P3 = ROOT / 'data' / 'parts1126'
    if not (P3 / '21' / 'target.stl').exists():
        t0 = time.perf_counter()
        g = subprocess.run([sys.executable, str(ROOT / 'tools' / 'make_jobs_parts.py'), str(P3), '21', '1126'],
                           capture_output=True, text=True, encoding='utf-8', errors='replace')
        print(f'연습 3 형상 21개 만듦: {time.perf_counter() - t0:.0f} s (계산 시간에 넣지 않음)')
        if g.returncode:
            print(g.stdout[-400:], g.stderr[-800:])
    fp = fingerprint(P3)
    print(f'형상 지문 {fp}:', '✅ 노트북에서 잰 연습 3 과 같은 형상' if fp == P3_FP
          else '⚠️ 노트북 형상과 다름(노트북과 시간 비교가 정확하지 않음)')
    SCENARIOS = sorted(d for d in P3.iterdir() if (d / 'target.stl').exists())

times, inner, plans, rcs = {}, {}, {}, {}
for scen in SCENARIOS:
    out = OUT / scen.name
    t0 = time.perf_counter()
    p = subprocess.run([sys.executable, str(ROOT / 'src' / 'solve.py'), str(scen), str(out)],
                       capture_output=True, text=True, encoding='utf-8', errors='replace')
    times[scen.name] = time.perf_counter() - t0
    rcs[scen.name] = p.returncode
    txt = (out / 'solve.txt').read_text(encoding='utf-8') if p.returncode == 0 and (out / 'solve.txt').exists() else ''
    m = re.search(r'^time\t.*\btotal=([\d.]+)s', txt, re.M)
    inner[scen.name] = float(m.group(1)) if m else None      # solve.py 가 잰 시간(프로그램 시작 약 1 s 제외)
    m = re.search(r'^best\t(\S+)', txt, re.M)
    plans[scen.name] = m.group(1) if m else '-'
    last = p.stdout.strip().splitlines()[-1] if p.returncode == 0 and p.stdout.strip() else p.stderr[-400:]
    print(f"{scen.name}: {times[scen.name]:6.1f} s  {'✅' if times[scen.name] <= LIMIT_S else '⛔ 5분 넘음'}  | {last}")

## 4. 채점 (계산 시간에 넣지 않음)

In [ ]:
import json
import pandas as pd

VAL = OUT / time.strftime('_val_%H%M%S')   # 채점기는 이미 차 있는 폴더에는 쓰지 않으므로, 돌릴 때마다 새 폴더에
rows, mks = [], {}
for scen in SCENARIOS:
    out = OUT / scen.name
    v = subprocess.run([sys.executable, '-m', 'waam_validator.cli', 'run', str(out),
                        '--output', str(VAL / scen.name), '--json'], capture_output=True, text=True)
    try:
        s = json.loads(v.stdout)
        mks[scen.name] = float(s['schedule']['makespan_s'])
        rows.append({'형상': scen.name, '계산 시간(s)': round(times[scen.name], 1),
                     '5분 안': '✅' if times[scen.name] <= LIMIT_S else '⛔',
                     'makespan(s)': round(float(s['schedule']['makespan_s']), 1), '판정': s['status'],
                     'IoU(%)': round(100 * s['shape']['iou'], 2), '충돌': s['collision']['collision_event_count']})
    except Exception:
        rows.append({'형상': scen.name, '계산 시간(s)': round(times[scen.name], 1), '판정': '채점 오류',
                     '메시지': (v.stderr or v.stdout)[-200:]})
df = pd.DataFrame(rows)
total = {'형상': '합계', '계산 시간(s)': round(df['계산 시간(s)'].sum(), 1)}
if 'makespan(s)' in df:
    total['makespan(s)'] = round(df['makespan(s)'].sum(), 1)
    total['판정'] = f"{(df['판정'] == 'PASS').sum()}/{len(df)} PASS"
pd.concat([df, pd.DataFrame([total])], ignore_index=True)

## 5. 캡처
4번 표(형상별 계산 시간·makespan·판정)와 1번의 컴퓨터 정보를 함께 캡처합니다.
형상마다 고른 계획과 단계별 시간(경로 계산, 계획, 국소 탐색)은 `/content/out/<형상>/solve.txt` 에 있습니다.

## 6. 대시보드 비교용 기록 내려받기
형상마다 계산 시간·makespan·고른 계획을 한 줄씩 적은 파일(`timing_colab_<묶음>.log`)을 내려받습니다.
이 파일을 주면 대시보드 '계산 시간' 화면에서 노트북 측정과 나란히 볼 수 있습니다.
줄마다 앞의 시간은 노트북 기록과 같은 기준인 `solve.py` 가 잰 시간(프로그램 시작 약 1 s 제외)이고,
끝의 `wall=` 은 4번 표의 계산 시간(프로그램 시작 포함)입니다.

In [ ]:
name = 'public' if SET.startswith('공개') else 'parts1126'
LOG = OUT / f'timing_colab_{name}.log'
ver = subprocess.run(['git', '-C', str(ROOT), 'log', '-1', '--format=%h'], capture_output=True, text=True).stdout.strip()
try:
    cpu = next(l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name'))
except Exception:
    cpu = '?'
lines = [f"# {SET} · 코랩 {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} · 우리 코드 {ver} · {cpu}"]
for scen in SCENARIOS:
    n = scen.name
    t = inner[n] if inner.get(n) is not None else times[n]
    lines.append(f"{n} 코랩  벽시계 {t:7.1f} s  makespan {mks.get(n, 0.0):.3f}  {plans[n]}  rc={rcs[n]}  wall={times[n]:.1f}")
LOG.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(LOG.read_text(encoding='utf-8'))
try:
    from google.colab import files
    files.download(str(LOG))
except ImportError:
    print('코랩이 아니라 내려받기는 건너뜀:', LOG)